# solveit-tex

A utility for creating LaTeX papers using SolveIT, to facilitate Human-Authored, Computer-Interrogated (HACI) writing workflows.

The idea is to be able to work in Solvit and see somewhat of what you're working on and be able to interrogate it via the AI incrementally and then have it generate a paper.

Full interoperability between Solve It and LaTeX isn't entirely something I'm going for, but I'm trying to make it so that things that render In one medium, render in the other to as high a degree as possible.

In [ ]:
#| export
import os, subprocess, json, re, sys
from pathlib import Path
from IPython.display import HTML, display

## Utilities called by other routines

In [ ]:
#| export
def get_private_url(path: str):
    "Get the private URL for a file on the solveit cloud instance"
    server = os.getenv('PRIVATE_DOMAIN')
    if not server: raise ValueError("PRIVATE_DOMAIN not set")
    path = os.path.abspath(path)
    return f"https://{server}.solve.it.com{path.replace('/app/data', '/static')}"

### Figures

In [ ]:
#| export
def parse_figure(line: str):
    r"""Parse markdown figure syntax with multiple images on one line: ![alt1](img1.png) ![alt2](img2.png)\{width=45% #fig:label}
     Images on one line get grouped into a single figure, with the final caption and label being the one used for the group"""
    import re
    
    # Look for escaped attributes at the end: \{...}
    attrs = ''
    attr_match = re.search(r'\\\{([^}]*)\}\s*$', line)
    if attr_match:
        attrs = attr_match.group(1)
        line = line[:attr_match.start()]  # Remove the attributes part
    
    # Find all image patterns on the line
    pattern = r'!\[([^\]]*)\]\(([^)]+)\)'
    matches = re.findall(pattern, line.strip())
    
    if not matches: return None
    
    images = []
    caption = ""
    
    for i, (alt, path) in enumerate(matches):
        img = {'path': path.strip()}
        images.append(img)
        
        # Last image sets the caption and label
        if i == len(matches) - 1:
            caption = alt
    
    # Extract width and label from attributes
    width_m = re.search(r'width=([^\s#]+)', attrs)
    label_m = re.search(r'#fig:([^\s}]+)', attrs)
    width = width_m.group(1) if width_m else None
    label = label_m.group(1) if label_m else None
    
    # Apply width to all images if specified
    if width:
        for img in images:
            img['width'] = width
    
    # If caption is just a filename, treat as no caption
    if caption:
        path_basenames = [Path(p['path']).name for p in images]
        if caption in path_basenames or caption in [p['path'] for p in images]:
            caption = ""
    
    return {'caption': caption, 'images': images, 'label': label}

In [ ]:
# Test it
s = r'![](images/mnist_ring_op_k_filmr_family.png) ![MNIST ring op$^k$ compositionality - FILMR \& Matrix Op](images/mnist_ring_op_k_mat_family.png)\{#fig:op_k}'
parse_figure(s)

{'caption': 'MNIST ring op$^k$ compositionality - FILMR \\& Matrix Op',
 'images': [{'path': 'images/mnist_ring_op_k_filmr_family.png'},
  {'path': 'images/mnist_ring_op_k_mat_family.png'}],
 'label': 'op_k'}

In [ ]:
#| export
def make_figure(fig_dict: dict):
    "Generate LaTeX figure environment from image specs."
    images, caption, label = fig_dict['images'],  fig_dict.get('caption', ''), fig_dict.get('label', '')
    lines = ['\\begin{figure}[htbp]', '\\centering']  # Start figure environment
    for img in images:
        width_opt = f'[width={img["width"]}]' if 'width' in img else '[width=\\linewidth]' # Add width if specified, defalt to linewidth
        lines.append(f'\\includegraphics{width_opt}{{{img["path"]}}}')  # Include the image
    if caption: lines.append(f'\\caption{{{caption}}}')  # Add caption if provided
    if label: lines.append(f'\\label{{fig:{label}}}')  # Add label if provided
    lines.append('\\end{figure}')  # Close figure environment
    return '\n'.join(lines)

In [ ]:
fig_tex = make_figure(im_dict)
fig_tex

'\\begin{figure}[htbp]\n\\centering\n\\includegraphics[width=\\linewidth]{images/mnist_ring_op_k_filmr_family.png}\n\\includegraphics[width=\\linewidth]{images/mnist_ring_op_k_mat_family.png}\n\\caption{MNIST ring op$^k$ compositionality - FILMR \\& Matrix Op}\n\\label{fig:op_k}\n\\end{figure}'

### Tables

Take a look at `~/hoplas-paper/hoplas-paper-memory/RESULTS_WIP.md`. There are a lot of tables in there that I could use for the results section of the paper I'm working on.
So we ought to have some way of converting markdown tables to LaTeX tables. And allowing for an option of label captions and reference numbers. How should we do that?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me look at the tables in that file first to see what we're working with:





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;file_view(&#x27;~/hoplas-paper/hoplas-paper-m…&quot;)→&quot;1: # hoplas — Results (WORK IN PROGRESS)…&quot;</code></summary>

```json
{
  "id": "call_ef139614688e438ea0fe5672",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "file_view('~/hoplas-paper/hoplas-paper-memory/RESULTS_WIP.md')"
    }
  },
  "result": "<TRUNCATED>…s — Results (WORK IN PROGRESS)\n2: \n3: > **Not vetted. Not for distribution.** Working draft. The KGE section is\n4: > auto-compiled from the autonomous campaign logs (W&B / `scripts/results.sh` /\n5: > `scripts/scores.sh`). The Ring section (Part I) is now filled from the op^k\n6: > composition study; the **Dihedral section (Part II) still needs work** (see the\n7: > `NEEDS DATA` marker — it's an identifiability result, not a metric table yet).\n8: > This file is intentionally **not** committed to git (would bloat the\n9: > distributable repo, esp. once images are added). Paste PNGs in freely while it\n10: > lives locally.\n11: \n12: **Arrows:** ↑ = higher is better, ↓ = lower is better.\n13: \n14: ---\n15: \n16: # Part I — Ring (cyclic group) task\n17: \n18: The core hoplas experiment: a `Projector` maps data into a latent space where a\n19: single learned operator advances each point to \"the next one in a cycle\"\n20: (`train_ops.py --target ring`). Self-supervised — similarity loss + SIGReg +\n21: reconstruction, no negatives. Datasets: synthetic `line` (clean ring) and\n22: `mnist` (VAE latents cycling `0→1→…→9→0`). See README §2.\n23: \n24: ## I.1 Supervised rotation benchmark (`train_simple_rot.py`)\n25: \n26: Sanity check / building block: can each operator represent a single fixed\n27: nd-rotation? Final MSE after training, sweeping `corr_nd` = # input channels\n28: sharing an inter-channel correlation (`corr=0.9`). *(Real numbers, from README.)*\n29: \n30: | run | nd=1 ↓ | nd=2 ↓ | nd=4 ↓ | nd=8 ↓ | nd=16 ↓ |\n31: |------------|---------:|---------:|---------:|---------:|---------:|\n32: | filmr      | 0.001216 | 0.000698 | 0.000496 | 0.002542 | 0.001237 |\n33: | filmr_expm | 0.000000 | 0.000001 | 0.000001 | 0.000008 | 0.000028 |\n34: | matop      | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000 |\n35: | matop2     | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000 |\n36: | ph_2       | 0.054844 | 0.055158 | 0.055196 | 0.047945 | 0.038888 |\n37: | ph_4       | 0.068751 | 0.067965 | 0.067233 | 0.061337 | 0.054137 |\n38: | ph_8       | 0.061031 | 0.060617 | 0.059060 | 0.054227 | 0.046926 |\n39: | ph_16      | 0.000000 | 0.000000 | 0.000001 | 0.000006 | 0.000010 |\n40: \n41: `ph_N` = `PHMLinear` at hypercomplex order N. Orders below nd^(2/3)=16 are\n42: genuinely constrained (non-zero structural floor); order 16 saturates the\n43: parameterization and solves the rotation exactly. Constrained orders improve as\n44: more input channels are correlated (loss drops left→right).\n45: \n46: ## I.2 Self-supervised ring task (`train_ops.py --target ring`)\n47: \n48: The headline result. After self-supervised training (no cycle-consistency term),\n49: we apply the learned operator **k times** to held-out latents, decode, and ask a\n50: pixel-space classifier `P(pred = (i+k) mod n)` — does composing the operator\n51: `op^k` trace out the cyclic group? Two readouts, both swept to **k=100** (eval:\n52: `scripts/score_transitions.py` + `transition_accuracy()`; MNIST n=10, chance=10%,\n53: clean classifier ceiling 99.72%):\n54: \n55: - **transition accuracy ↑** — fraction landing on the correct `(i+k) mod n` class.\n56: - **output diversity ↑** — effective # classes among the decoded outputs (10 =\n57:   uniform, 1 = collapsed to a single digit). *This is the anti-collapse probe.*\n58: \n59: The result is an **operator-type × unit-norm grid**, and the punchline is that\n60: for the *structured* operators (ph, quat) **turning unit-norm OFF is dramatically\n61: better** — the cyclic algebra wants the embedding geometry *free* (a multi-lobed\n62: star), not pinned to a sphere. This is \"ring algebra without ring geometry\": the\n63: group structure lives in the **operator**, not the latent metric.\n64: \n65: | operator family | best variant | step-1 acc ↑ | closure peaks (k≡0 mod n)? | acc @ k=100 ↑ | diversity @ k=100 ↑ | verdict |\n66: |---|---|---:|:--:|---:|---:|---|\n67: | **PH** (hypercomplex) | `ph_8_nonorm` | ~93% | **yes, to k=100** | **~90%** | **~9–10** | **composes + stays diverse** |\n68: | **quat** (Hamilton/dual) | `kquat_nonorm`, `kdualquat_nonorm` | ~85% | yes | ~65–80% | ~8–9 | composes (right-structured hypercomplex) |\n69: | **filmr_expm** (rigid rotation) | `filmr_expm_*_norm` | ~97% | razor-sharp at k≡0, **chance in between** | spikes only | osc. 5–10; `nonorm`→1 by k≈50 | closes at full revolutions, hollow between |\n70: | **matop** (unconstrained) | `matop_clip_nonorm` | ~97% | **none — never closes** | ~chance | collapses to ~3–4 | decays to chance, top-eigenspace collapse |\n71: \n72: Key reads from the curves (figures below):\n73: \n74: - **PH `nonorm` is the clean winner.** `ph_8_nonorm` holds ~90% transition accuracy\n75:   *and* near-full diversity all the way to k=100, with closure bumps still firing\n76:   after 10 revolutions — emergent, never imposed. `nonorm ≫ norm` for PH; the worst\n77:   PH is `ph_4_norm` (decays to ~chance by k=100). **Unit-norm hurts the structured op.**\n78: - **quat** (a right-structured hypercomplex) also c…</TRUNCATED>"
}
```

</details>







Looking at the tables, they're standard markdown with alignment markers (`:---`, `---:`, `:---:`). I see a few design options for labels and captions:

**Option 1: Attributes block (like figures)**
```markdown
| col1 | col2 |
|------|------|
| data | data |
\{#tab:label caption="Table caption here"}
```

**Option 2: Separate caption line**
```markdown
| col1 | col2 |
|------|------|
| data | data |
\{#tab:label}
*Caption text here*
```

**Option 3: Inline caption after table**
```markdown
| col1 | col2 |
|------|------|
| data | data |
*Caption text here* \{#tab:label}
```

Option 1 feels most consistent with how we handle figures. The LaTeX output would be a `table` environment with `\caption{}` and `\label{}`.

For alignment, we'd parse the separator row to detect left/center/right and generate the appropriate column spec (`l`, `c`, `r`).

Which approach appeals to you? Or would you prefer something different?

Ooo I like 3. And in fact, I was lamenting the fact that for our figures, you couldn't see the figure captions on Solvit because they weren't rendering. So, I might actually want to change the figures thing to support option three as well, but let's just do the tables first.  So I guess the export script is going to need to detect when it's starting to see a table, and when it's finished with a table, and when it's finally got the caption line?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right, so we'd need a state machine that:

1. Sees `|` at line start → start collecting table rows
2. Keeps collecting while lines start with `|` (including the separator row)
3. On the next line, check if it matches the caption pattern: `*text*` optionally followed by `\{#tab:label}`
4. If yes, attach caption to table; if no, table is done

We'd want a `parse_table()` function similar to `parse_figure()`, returning something like:
```python
{'headers': [...], 'rows': [[...], ...], 'alignments': ['l', 'c', 'r', ...], 'caption': '...', 'label': '...'}
```

Then `make_table()` would generate the LaTeX `table` environment with proper column specs.

Shall we write `parse_table()` first?

Sure, sounds good.  Try to keep it as short as possible, at least in terms of number of vertical lines. You can do long one-liners if you want.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
def parse_table(lines: list[str]):
    """Parse markdown table with optional caption. Returns (table_dict, lines_consumed) or (None, 0)."""
    if not lines or not lines[0].startswith('|'): return None, 0
    i = 0
    while i < len(lines) and lines[i].startswith('|'): i += 1
    if i < 3: return None, 0
    sep = [c.strip() for c in lines[1].split('|')[1:-1]]
    aligns = ['c' if c.startswith(':') and c.endswith(':') else 'r' if c.endswith(':') else 'l' for c in sep]
    headers = [c.strip() for c in lines[0].split('|')[1:-1]]
    rows = [[c.strip() for c in r.split('|')[1:-1]] for r in lines[2:i]]
    caption, label = '', None
    if i < len(lines):
        m = re.match(r'\*([^*]+)\*(?:\s*\\\{#tab:([^}]+)\})?', lines[i])
        if m: caption, label = m.group(1), m.group(2); i += 1
    return {'headers': headers, 'rows': rows, 'alignments': aligns, 'caption': caption, 'label': label}, i
```

In [ ]:
def parse_table(lines):
    """Parse markdown table with optional caption. Returns (table_dict, lines_consumed) or (None, 0)."""
    if isinstance(lines, str): lines = lines.split('\n')
    if not lines or not lines[0].startswith('|'): return None, 0
    i = 0
    while i < len(lines) and lines[i].startswith('|'): i += 1
    if i < 3: return None, 0
    sep = [c.strip() for c in lines[1].split('|')[1:-1]]
    aligns = ['c' if c.startswith(':') and c.endswith(':') else 'r' if c.endswith(':') else 'l' for c in sep]
    headers = [c.strip() for c in lines[0].split('|')[1:-1]]
    rows = [[c.strip() for c in r.split('|')[1:-1]] for r in lines[2:i]]
    caption, label = '', None
    if i < len(lines):
        m = re.match(r'\s*\*([^*]+)\*(?:\s*\\\{#tab:([^}]+)\})?', lines[i])
        if m: caption, label = m.group(1), m.group(2); i += 1
    return {'headers': headers, 'rows': rows, 'alignments': aligns, 'caption': caption, 'label': label}, i


## Conversion script for .ipynb -> .tex

In [ ]:
# test it
test_str = """
| lambda-neg | TEST MRR ↑ | H@1 ↑ | H@3 ↑ | H@10 ↑ | MR(cos) ↓ |
|---|---|---|---|---|---|
| 0 | 0.457 | 0.395 | 0.487 | 0.573 | ~1300 |
| 0.003 | 0.4645 | 0.402 | 0.497 | 0.576 | 1236 |
| 0.01 | 0.4746 | 0.414 | 0.504 | 0.585 | 1233 |
| 0.03 | 0.489 | 0.431 | 0.519 | 0.592 | **1180** |
| 0.20 (champion) | **0.5151** | **0.4665** | **0.5327** | **0.6151** | 1749 |
 *The contrastive lever — `--lambda-neg` sweep (WN18RR, order2 nd512 lambd0.10)* \{#tab:contr}
"""
out = parse_table(test_str) 
out

(None, 0)

In [ ]:
#| export
def export_ipynb_to_tex(ipynb_path: str, output_path: str = None):
    r"""Export a Solveit dialog (.ipynb) to a compilable LaTeX file.
    Cells are emitted in document order, each preceded by a `% <cell-id>` comment.
    The `## Abstract` cell emits `\begin{document}`, dividing preamble from document body."""

    ipynb_path = os.path.expanduser(ipynb_path)
    output_path = os.path.expanduser(output_path) if output_path else Path(ipynb_path).with_suffix('.tex')

    nb = json.loads(Path(ipynb_path).read_text())
    out = []

    for cell in nb['cells']:
        content = ''.join(cell['source'])

        if '#| export' not in content:
            continue

        filtered = '\n'.join(l for l in content.split('\n') if not l.startswith('#| '))
        out.append(f'% {cell["id"]}')

        if cell['cell_type'] == 'raw':
            out.append(filtered)
            continue

        lines = filtered.split('\n')
        i = 0
        while i < len(lines):
            line = lines[i]

            if line.startswith('# ') and not line.startswith('## '):
                out.append(f'\\title{{{line[2:].strip()}}}\n')
            elif line.startswith('\\author{'):
                out.append(line)
                while i < len(lines) and not lines[i].strip().endswith('}'):
                    i += 1
                    if i < len(lines):
                        out.append(lines[i])
            elif line == '## Abstract':
                out.append('\\begin{document}\n\n\\maketitle\n')
                out.append('\\begin{abstract}\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    out.append(lines[i])
                    i += 1
                out.append('\\end{abstract}\n')
                continue
            elif line == '## References':
                out.append('\\small\n')
                bib_match = re.search(r'(\w+)\.bib', content)
                if bib_match:
                    out.append(f'\\bibliographystyle{{unsrt}}\n\\bibliography{{{bib_match.group(1)}}}\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    i += 1
                continue
            elif line.startswith('### '):
                out.append(f'\\subsection{{{line[4:].strip()}}}\n')
            elif line.startswith('## '):
                out.append(f'\\section{{{line[3:].strip()}}}\n')
            else:
                fig_dict = parse_figure(line)
                if fig_dict:
                    out.append(make_figure(fig_dict))
                else:
                    out.append(line)

            i += 1

    final = '\\documentclass{article}\n\\usepackage{graphicx}\n\n'
    final += '\n'.join(out) + '\n\n'
    final += '\\end{document}\n'
    Path(output_path).write_text(final)
    print(f'Created {output_path}')
    output_url = get_private_url(output_path)
    display(HTML(f'<a href="{output_url}" target="_blank">{output_url}</a>'))

In [ ]:
export_ipynb_to_tex('example/neurips_2026.ipynb')

Created example/neurips_2026.tex


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex</a>)

## Conversion script for tex -> pdf

In [ ]:
#| export
def compile_latex(tex_file: str, cwd: str = '.'):
    "Run full LaTeX compilation: pdflatex → bibtex → pdflatex → pdflatex"

    cwd = os.path.expanduser(cwd)
    tex_file = os.path.expanduser(tex_file)
    
    # Make path absolute before splitting
    full_tex_path = os.path.abspath(os.path.join(cwd, tex_file))
    work_dir = os.path.dirname(full_tex_path)
    
    base_name = os.path.splitext(os.path.basename(full_tex_path))[0]
    tex_content = Path(full_tex_path).read_text()
    bib_match = re.search(r'\\bibliography\{([^}]+)\}', tex_content)
    
    print("Running pdflatex (pass 1)...")
    r1 = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                   shell=True, cwd=work_dir, capture_output=True, text=True)
    if r1.returncode != 0:
        print(f"  FAILED:\n{r1.stdout[-3000:]}")
        return
    
    if bib_match:
        bib_name = bib_match.group(1)
        print(f"\nRunning bibtex ({bib_name})...")
        r = subprocess.run(f'bibtex {base_name}',
                          shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout}")
            return
    
    for i in [2, 3]:
        print(f"\nRunning pdflatex (pass {i})...")
        r = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                       shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout[-3000:]}")
            return

    pdf_path = full_tex_path.replace('.tex', '.pdf') 
    pdf_url = get_private_url(pdf_path)
    print(f"\nSuccess!")
    print(f"File: {pdf_path}")
    print(f"PDF url: {pdf_url}")
    sys.stdout.flush() 
    display(HTML(f'<a href="{pdf_url}" target="_blank">{pdf_url}</a>'))

In [ ]:
compile_latex('example/neurips_2026.tex')

Running pdflatex (pass 1)...



Running bibtex (references)...



Running pdflatex (pass 2)...



Running pdflatex (pass 3)...



Success!


File: /app/data/solveit-tex/example/neurips_2026.pdf


PDF url: https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf</a>)

## All-in-One Wrapper Function

In [ ]:
#| export
async def current_to_pdf():
    """
    Wrapper that converts the current dialogue to PDF and prints the private URL for it.
    Usage: await current_to_pdf()
    """
    from dialoghelper.solveitskill import curr_dialog, realpath

    name = (await curr_dialog())['name']
    path = f'{await realpath("/")}/{name}.ipynb'
    export_ipynb_to_tex(path)
    display(HTML(f'<br>'))
    compile_latex(path.replace('.ipynb', '.tex'))

## Allow LLM to call routines herein

In [ ]:
#| export
from pyskills import allow 

allow(export_ipynb_to_tex)
allow(current_to_pdf)
allow(compile_latex) 

<function __main__.compile_latex(tex_file: str, cwd: str = '.')>